# Elife Lung Cancer Non Coding Biomarker Random Forest
Andrew E. Davidson aedaivds@ucsc.edu 03/04/25

Copyright (c) 2020-2023, Regents of the University of California All rights reserved. https://polyformproject.org/licenses/noncommercial/1.0.0

1. Train a Random Forest using all the non-coding genes from the Lung Cancer', 'Healthy donor' samples
2. Use feature Importance to identify set of best features


ref:  
- intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/createNonCodingDataSet.ipynb
- intraExtraRNA_POC/jupyterNotebooks/elife/elifeBinaryRandomForestResults.ipynb

In [1]:
import ipynbname

# use display() to print an html version of a data frame
# useful if dataFrame output is not generated by last like of cell
from IPython.display import display

import joblib
import math
import numpy as np
import os
import pandas as pd
import pprint as pp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import sys

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

#outDir = f'{notebookDir}/{notebookName}.out'
outDir = f'/private/groups/kimlab/aedavids/elife/{notebookName}.out'
os.makedirs(outDir, exist_ok=True)
print(f'outDir:\n{outDir}')

dataOutDir = os.path.join(outDir, "data")
os.makedirs(dataOutDir, exist_ok=True)
print(f'\ndataOutDir ;\n{dataOutDir}')

import logging
#loglevel = "DEBUG"
#loglevel = "INFO"
loglevel = "WARN"
# logFMT = "%(asctime)s %(levelname)s [thr:%(threadName)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logFMT = "%(asctime)s %(levelname)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logging.basicConfig(format=logFMT, level=loglevel)    
logger = logging.getLogger(notebookName)

meaningOfLife = 42

outDir:
/private/groups/kimlab/aedavids/elife/elifeLungCancerNon-CodingBiomarkerRandomForest.out

dataOutDir ;
/private/groups/kimlab/aedavids/elife/elifeLungCancerNon-CodingBiomarkerRandomForest.out/data


In [2]:
# setting the python path allows us to run python scripts from using
# the CLI. 
ORIG_PYTHONPATH = os.environ['PYTHONPATH']

deconvolutionModules = notebookPath.parent.joinpath("../../../../deconvolutionAnalysis/python/")
print("deconvolutionModules: {}\n".format(deconvolutionModules))

PYTHONPATH = ORIG_PYTHONPATH + f':{deconvolutionModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../../python/src")
print("intraExtraRNA_POCModules: {}\n".format(intraExtraRNA_POCModules))

PYTHONPATH = PYTHONPATH + f':{intraExtraRNA_POCModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

########
os.environ["PYTHONPATH"] = PYTHONPATH
PYTHONPATH = os.environ["PYTHONPATH"]
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# to be able to import our local python files we need to set the sys.path
# https://stackoverflow.com/a/50155834
sys.path.append( str(deconvolutionModules) )
sys.path.append( str(intraExtraRNA_POCModules) )
#print("\nsys.path:\n{}\n".format(sys.path))

deconvolutionModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python

intraExtraRNA_POCModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python:/private/home/aedavid

In [3]:
from intraExtraRNA.elifeUtilities import loadMetaData

# Data Prep
Load count and meta data.  
Fit a Label Encoder

In [4]:
annotated_elife_lung_norm_counts_path = "/private/groups/kimlab/aedavids/elife/createNonCodingDataSet.out/data/annotated_elife_lung_norm_counts_2023-05-18.csv"
annotatedDF = pd.read_csv( annotated_elife_lung_norm_counts_path, index_col='gene')

In [5]:
print( annotatedDF.shape )
annotatedDF.iloc[0:5, 0:5]

(56607, 79)


,gene_biotype,SRR14506690,SRR14506691,SRR14506692,SRR14506693
gene,,,,,
(A)n,Microsatellite,148.057125,121.600515,551.413204,45.545968
(AAA)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAC)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAG)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAT)n,Microsatellite,0.000000,0.000000,0.000000,0.000000


In [6]:
byCol = 1
elifeLungCountsDF = annotatedDF.drop('gene_biotype', axis=byCol)
print( elifeLungCountsDF.shape )
#elifeLungCountsDF.iloc[0:5, 0:5]

(56607, 78)


In [7]:
# random forest fit expects count data to be samples x genes
elifeLungCountsDF = elifeLungCountsDF.transpose()
print(elifeLungCountsDF.shape)
elifeLungCountsDF.iloc[0:5, 0:5]

(78, 56607)


gene,(A)n,(AAA)n,(AAAAAAC)n,(AAAAAAG)n,(AAAAAAT)n
SRR14506690,148.057125,0.0,0.0,0.0,0.0
SRR14506691,121.600515,0.0,0.0,0.0,0.0
SRR14506692,551.413204,0.0,0.0,0.0,0.0
SRR14506693,45.545968,0.0,0.0,0.0,0.0
SRR14506694,86.068765,0.0,0.0,0.0,0.0


In [8]:
elifeMetaDF = loadMetaData()
selelectLungRows = elifeMetaDF['diagnosis'].isin( ['Lung Cancer', 'Healthy donor'] )
lungSampleIdDF = elifeMetaDF.loc[ selelectLungRows, :]
lungSampleIdDF

,sample_id,diagnosis
31,SRR14506690,Lung Cancer
32,SRR14506691,Lung Cancer
33,SRR14506692,Lung Cancer
34,SRR14506693,Lung Cancer
35,SRR14506694,Lung Cancer
...,...,...
219,SRR14506884,Healthy donor
220,SRR14506885,Healthy donor
221,SRR14506886,Healthy donor
222,SRR14506887,Healthy donor


In [9]:
# we can not control which class is labled zero
labelEncoder = LabelEncoder()
labelEncoder.fit( lungSampleIdDF.loc[:, 'diagnosis'])
print( labelEncoder.classes_)
yNP = labelEncoder.transform( lungSampleIdDF.loc[:, 'diagnosis'] )
yNP

['Healthy donor' 'Lung Cancer']


array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

# Train Model

In [10]:
%%time
rfModel = RandomForestClassifier(random_state=meaningOfLife) # , **kwags
rfModel.fit(elifeLungCountsDF, yNP)

CPU times: user 516 ms, sys: 4.09 ms, total: 520 ms
Wall time: 519 ms


RandomForestClassifier(random_state=42)

# Evaluate Model

In [11]:
def calculateAccuracy( 
            rfModel : RandomForestClassifier,
            xDF : pd.DataFrame,
            yNP : np.array ) :
    '''
    todo
    '''
    predictions = rfModel.predict( xDF )
    n = len(yNP)
    accuracy = sum( predictions == yNP ) / n

    return accuracy
    
accuracy = calculateAccuracy(rfModel , elifeLungCountsDF, yNP)
print( f'accuracy : {accuracy * 100}%' )

accuracy : 100.0%


In [12]:
# calculate the area under the curve

# predict_proba returns 2 probablities for each sample. control, treatment
pp = rfModel.predict_proba(elifeLungCountsDF)[:,1]
auc = roc_auc_score(yNP, pp)
print( f'roc : {auc}' )

roc : 0.9999999999999999


In [13]:
def selectImportantFeatures( 
        model : RandomForestClassifier,
        annotatedDF : pd.DataFrame,
        XDF : pd.DataFrame ) -> pd.DataFrame :
    '''
    returns a data frame with features that have non zero importance
    sorted in descending order by importance

    importance is a value between 0 and 1. The values are normalize

    example output
                      name  importance    gene_bioType
    451  ENSG00000277194.1    0.014160  Microsatellite
    489  ENSG00000285909.1    0.011819  Microsatellite
    542  ENSG00000289601.1    0.010526  Microsatellite
    231  ENSG00000237550.6    0.010144  Microsatellite
    381  ENSG00000264769.1    0.009759  Microsatellite
    
    annotatedDF is a count data frame with a 'gene_biotype' column
    '''
    # create a knock out vector
    nonZero = model.feature_importances_ > 0.0
    
    importances = model.feature_importances_[nonZero]
    featureNames = XDF.columns[nonZero]
    retDF = pd.DataFrame( {
                            'name' : featureNames,
                            'importance' : importances
                        })
    
    retDF.sort_values(by="importance", ascending=False, inplace=True)

    selectImportantRows = annotatedDF.index.isin( retDF.loc[:, 'name'] )
    importGenBioTypeSeries = annotatedDF.loc[selectImportantRows, 'gene_biotype']

    retDF['gene_bioType'] = importGenBioTypeSeries.values

    return retDF

###################
featureImportanceDF = selectImportantFeatures( rfModel, annotatedDF, elifeLungCountsDF )
print('featureImportanceDF.shape :', featureImportanceDF.shape)
featureImportanceDF.head()

featureImportanceDF.shape : (631, 3)


,name,importance,gene_bioType
451,ENSG00000277194.1,0.014160,Microsatellite
489,ENSG00000285909.1,0.011819,Microsatellite
542,ENSG00000289601.1,0.010526,Microsatellite
231,ENSG00000237550.6,0.010144,Microsatellite
381,ENSG00000264769.1,0.009759,Microsatellite


In [14]:
importGenBioTypeSeries = featureImportanceDF["gene_bioType"]
importGenBioTypeSeries.groupby(importGenBioTypeSeries).count()

gene_bioType
DNA                10
LINE               15
LTR                48
Microsatellite     26
Other             204
SINE                7
lncRNA            321
Name: gene_bioType, dtype: int64

## <span style="color:red;background-color:yellow">Explore features with high importance and biotype = 'Other'
Not sure what "Other" means. Alex mentioned several biotypes where lumped but was not sure what the components where. It looks like only 27 candidate features have an importance > the most important other

In [15]:
selectTypeOtherRows = featureImportanceDF["gene_bioType"] == "Other"
featureImportanceDF.loc[ selectTypeOtherRows, :].sort_values(by="importance", ascending=False).head()

,name,importance,gene_bioType
319,ENSG00000254659.3,0.004388,Other
129,ENSG00000224616.4,0.004379,Other
294,ENSG00000249898.9,0.004135,Other
486,ENSG00000285774.4,0.004083,Other
119,ENSG00000223704.2,0.004043,Other


In [16]:
featureImportanceDF.sort_values(by="importance", ascending=False)

,name,importance,gene_bioType
451,ENSG00000277194.1,0.014160,Microsatellite
489,ENSG00000285909.1,0.011819,Microsatellite
542,ENSG00000289601.1,0.010526,Microsatellite
231,ENSG00000237550.6,0.010144,Microsatellite
381,ENSG00000264769.1,0.009759,Microsatellite
...,...,...,...
546,GSATII,0.000158,Other
356,ENSG00000259419.2,0.000157,Other
245,ENSG00000241755.1,0.000154,Other
515,ENSG00000287682.1,0.000152,Other


In [17]:
selectImportantRows = featureImportanceDF["importance"] > 0.004388
aedwipDF = featureImportanceDF.loc[selectImportantRows, :]
print(aedwipDF.shape)
xxxSeries = aedwipDF["gene_bioType"]
xxxSeries.groupby(xxxSeries).count()

(27, 3)


gene_bioType
Microsatellite    26
Other              1
Name: gene_bioType, dtype: int64

# Hyperparameter Tuning

In [18]:
%%time
def randomForestTrainer(annotatedDF, XDF, yNP):
    '''
    1. create and fit a random forest model
    2. calculate accuracy and AUC
    3. find important features (features with importance > 0)
    4. for each gene_biotype count the number of important genes
    '''
    retDict = {}
    
    model = RandomForestClassifier(random_state=meaningOfLife) # , **kwags
    model.fit(XDF, yNP)
    retDict['model'] = model
    
    accuracy = calculateAccuracy(model, XDF, yNP)
    retDict['accuracy'] = accuracy
    
    # calculate the area under the curve
    # predict_proba returns 2 probablities for each sample. control, treatment
    pp = model.predict_proba(XDF)[:,1]
    auc = roc_auc_score(yNP, pp)
    retDict['auc'] = auc

    
    featureImportanceDF = selectImportantFeatures( model, annotatedDF, XDF )
    retDict['featureImportanceDF'] = featureImportanceDF

    importantGenBioTypeSeries = featureImportanceDF["gene_bioType"]
    importantBioTypeCountsSeries = importantGenBioTypeSeries.groupby(importantGenBioTypeSeries).count()
    retDict['importantBioTypeCountsSeries'] = importantBioTypeCountsSeries

    return retDict

itteration1ResultsDict = randomForestTrainer(annotatedDF, elifeLungCountsDF, yNP)

CPU times: user 1.11 s, sys: 52.1 ms, total: 1.16 s
Wall time: 1.16 s


In [19]:
print( itteration1ResultsDict.keys() )

dict_keys(['model', 'accuracy', 'auc', 'featureImportanceDF', 'importantBioTypeCountsSeries'])


In [20]:
importantBioTypeCountsSeries = itteration1ResultsDict['importantBioTypeCountsSeries']
print('importantBioTypeCountsSeries')
display(importantBioTypeCountsSeries)

importantBioTypeCountsSeries


gene_bioType
DNA                10
LINE               15
LTR                48
Microsatellite     26
Other             204
SINE                7
lncRNA            321
Name: gene_bioType, dtype: int64

In [21]:
%%time
def loop():

    retDict = {}

    # boot strap, train on all features
    resultsDict = randomForestTrainer(annotatedDF, elifeLungCountsDF, yNP)
    i = 0    
    retDict[i] = resultsDict

    # python does not have a native do while loop. use while :true and break
    while True:
        print('\n******')
        featureImportanceDF = resultsDict['featureImportanceDF']
        print(f'i : {i} featureImportanceDF.shape : {featureImportanceDF.shape}')
        nFeatures = featureImportanceDF.shape[0]

        print(f' accuracy : {resultsDict["accuracy"]}% auc : {resultsDict["auc"]}')
        
        if nFeatures <= 20:
            print(f'finished nFeatures : {nFeatures}')
            break

        allImportantFeatures = featureImportanceDF.loc[:,'name']
        # reduced the number of features in half. Take the top,
        # most important features
        nFeatures = int( allImportantFeatures.shape[0] / 2 )
        print(f'nFeatures : {nFeatures}')
        features = allImportantFeatures[0:nFeatures]
        XDF = elifeLungCountsDF.loc[:, features]
        print(f'XDF.shape : {XDF.shape}')

        i += 1
        resultsDict = randomForestTrainer(annotatedDF, XDF, yNP)
        retDict[i] = resultsDict
        
        if i > 10:
            # this should never happen
            logger.error('i = {i} > 10 this should never happend')
            break

    return retDict

loopResultsDict = loop()


******
i : 0 featureImportanceDF.shape : (631, 3)
 accuracy : 1.0% auc : 0.9999999999999999
nFeatures : 315
XDF.shape : (78, 315)

******
i : 1 featureImportanceDF.shape : (234, 3)
 accuracy : 1.0% auc : 1.0
nFeatures : 117
XDF.shape : (78, 117)

******
i : 2 featureImportanceDF.shape : (110, 3)
 accuracy : 1.0% auc : 1.0
nFeatures : 55
XDF.shape : (78, 55)

******
i : 3 featureImportanceDF.shape : (55, 3)
 accuracy : 1.0% auc : 1.0
nFeatures : 27
XDF.shape : (78, 27)

******
i : 4 featureImportanceDF.shape : (27, 3)
 accuracy : 1.0% auc : 1.0
nFeatures : 13
XDF.shape : (78, 13)

******
i : 5 featureImportanceDF.shape : (13, 3)
 accuracy : 1.0% auc : 1.0
finished nFeatures : 13
CPU times: user 2.05 s, sys: 71.8 ms, total: 2.13 s
Wall time: 2.12 s


In [22]:
i5Results = loopResultsDict[5]
i5Results

{'model': RandomForestClassifier(random_state=42),
 'accuracy': np.float64(1.0),
 'auc': np.float64(1.0),
 'featureImportanceDF':                   name  importance gene_bioType
 0    ENSG00000237550.6    0.146746         SINE
 1    ENSG00000235078.1    0.118225       lncRNA
 4    ENSG00000264769.1    0.095012       lncRNA
 5    ENSG00000234741.9    0.094382       lncRNA
 8    ENSG00000280800.1    0.089595       lncRNA
 6    ENSG00000259278.2    0.076215       lncRNA
 3    ENSG00000285909.1    0.075287        Other
 2   ENSG00000216863.10    0.072508       lncRNA
 9    ENSG00000276141.4    0.065132       lncRNA
 7       HERVIP10FH-int    0.048111        Other
 10  ENSG00000235531.11    0.041690       lncRNA
 11   ENSG00000198106.8    0.038631       lncRNA
 12              AluSx3    0.038466          LTR,
 'importantBioTypeCountsSeries': gene_bioType
 LTR       1
 Other     2
 SINE      1
 lncRNA    9
 Name: gene_bioType, dtype: int64}